## MASTER ACTIVATOR TEMPLATE

In [ ]:
import sys, os

_stale = ['chromadb','gradio','sentence_transformers', 'pydantic',
          'huggingface_hub','langchain','transformers', 'albumentations']
for m in list(sys.modules.keys()):
    if any(s in m for s in _stale):
        del sys.modules[m]

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# All project paths
LIB_DIR  = "/content/drive/MyDrive/radiology_ai/libs"
DATA_DIR = "/content/drive/MyDrive/radiology_ai/data"
MDL_DIR  = "/content/drive/MyDrive/radiology_ai/models"
RES_DIR  = "/content/drive/MyDrive/radiology_ai/results"
RAG_DIR  = "/content/drive/MyDrive/radiology_ai/rag_papers"
CHR_DIR  = "/content/drive/MyDrive/radiology_ai/chroma_db"

for d in [LIB_DIR,DATA_DIR,MDL_DIR,RES_DIR,RAG_DIR,CHR_DIR]:
    os.makedirs(d, exist_ok=True)

if LIB_DIR in sys.path: sys.path.remove(LIB_DIR)
sys.path.insert(0, LIB_DIR)

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Libs loaded | Device: {DEVICE.upper()}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("No GPU — go to Runtime → Change runtime type → T4 GPU")

## PyTorch Dataset + DataLoaders

In [ ]:
# Data Loder
import torch, cv2, os, json
import numpy as np, pandas as pd
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

# --- 1. CONFIGURATION ---
DATA_DIR = "/content/drive/MyDrive/radiology_ai/data"
DISEASES = json.load(open(f"{DATA_DIR}/diseases.json"))

# Medical Image Augmentation Strategy
TRAIN_AUG = A.Compose([
    A.Resize(256, 256),
    A.RandomCrop(224, 224),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, p=0.4),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.4),
    A.CLAHE(clip_limit=3.0, p=0.4), # Crucial for X-ray contrast
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

VAL_AUG = A.Compose([
    A.Resize(256, 256),
    A.CenterCrop(224, 224),
    A.CLAHE(clip_limit=3.0, p=1.0), # Apply to validation for consistency
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# --- 2. DATASET CLASS ---
class NIHDataset(Dataset):
    def __init__(self, csv_path, transform=None, diseases=DISEASES):
        self.df = pd.read_csv(csv_path)
        self.transform = transform
        self.diseases = diseases

        # Verify the file_path column exists from our EDA step
        if 'file_path' not in self.df.columns:
            # Emergency fallback if Cell 4 wasn't run with absolute paths
            img_base = "/content/drive/MyDrive/radiology_ai/data/nih/images"
            self.df['file_path'] = self.df['Image Index'].apply(lambda x: os.path.join(img_base, x))

        print(f"📦 Dataset loaded from {os.path.basename(csv_path)}: {len(self.df)} images")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row['file_path']

        # Load image (BGR to RGB)
        img = cv2.imread(img_path)
        if img is None:
            # Return a zero tensor if image is missing to prevent crash
            return torch.zeros((3, 224, 224)), torch.zeros(len(self.diseases)), row['Image Index']

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Apply Albumentations
        if self.transform:
            img = self.transform(image=img)['image']

        # Convert pathology labels to tensor
        label = torch.tensor([row[d] for d in self.diseases], dtype=torch.float32)

        return img, label, row['Image Index']

# --- 3. CREATE DATALOADERS ---
print("Initializing DataLoaders...")

train_ds = NIHDataset(f"{DATA_DIR}/train.csv", transform=TRAIN_AUG)
val_ds   = NIHDataset(f"{DATA_DIR}/val.csv",   transform=VAL_AUG)
test_ds  = NIHDataset(f"{DATA_DIR}/test.csv",  transform=VAL_AUG)

# Recommended Batch Sizes for T4 GPU
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print(f"\n DataLoaders Ready!")
print(f"   Train: {len(train_loader)} batches")
print(f"   Val:   {len(val_loader)} batches")

## Load Fine-Tuned Model

In [ ]:
import torch, torch.nn as nn
import torchxrayvision as xrv

class FineTunedDenseNet(nn.Module):
    def __init__(self, num_classes=14):
        super().__init__()
        base = xrv.models.DenseNet(weights="densenet121-res224-all")
        self.features = base.features

        # This classifier includes the Pooling and Flattening layers internally
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.BatchNorm1d(1024),
            nn.Dropout(p=0.4),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(512, num_classes)
        )

        # This is the "Unexpected Key" from your error—it must be here!
        self.gradcam_layer = self.features.denseblock4

    def forward(self, x):
        if x.shape[1] == 3:
            x = x.mean(dim=1, keepdim=True)

        x = (x * 2048) - 1024
        feat = self.features(x)
        feat = torch.relu(feat)
        return self.classifier(feat)

##  Full Test Evaluation

In [ ]:
# Evaluate fine-tuned model on test set
import torch, json, os, numpy as np
import pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import (roc_auc_score, f1_score, precision_score,
                               recall_score, roc_curve, average_precision_score)
from sklearn.calibration import calibration_curve
from tqdm import tqdm

DISEASES = json.load(open(f"{DATA_DIR}/diseases.json"))

# Load fine-tuned model
ckpt = torch.load(f"{MDL_DIR}/finetuned_densenet121_best.pth", map_location=DEVICE, weights_only=False)
model = FineTunedDenseNet(len(DISEASES)).to(DEVICE)
model.load_state_dict(ckpt['model_state_dict'], strict=False)
model.eval()
print(f"Loaded fine-tuned model (val AUC={ckpt['best_val_auc']:.4f}, epoch={ckpt['epoch']})")

# Run on test set
all_labels, all_probs = [], []
with torch.no_grad():
    for imgs, labels, _ in tqdm(test_loader, desc="Evaluating test set"):
        logits = model(imgs.to(DEVICE))
        all_labels.append(labels.numpy())
        all_probs.append(torch.sigmoid(logits).cpu().numpy())

all_labels = np.vstack(all_labels)
all_probs  = np.vstack(all_probs)
all_preds  = (all_probs >= 0.5).astype(int)

# Per-class metrics
metrics = {}
for i, d in enumerate(DISEASES):
    y_true = all_labels[:,i]; y_prob = all_probs[:,i]; y_pred = all_preds[:,i]
    if y_true.sum() == 0: continue
    metrics[d] = {
        'AUC'      : roc_auc_score(y_true, y_prob),
        'F1'       : f1_score(y_true, y_pred, zero_division=0),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall'   : recall_score(y_true, y_pred, zero_division=0),
        'AP'       : average_precision_score(y_true, y_prob),
        'Positives': int(y_true.sum())
    }

mean_auc = np.mean([m['AUC'] for m in metrics.values()])
mean_f1  = np.mean([m['F1']  for m in metrics.values()])

# Print results table
print(f"\n{'='*60}")
print(f"TEST SET RESULTS — Fine-Tuned DenseNet121")
print(f"{'='*60}")
print(f"{'Disease':22} {'AUC':>6} {'F1':>6} {'Prec':>6} {'Rec':>6} {'N':>6}")
print("-"*60)
for d,m in sorted(metrics.items(), key=lambda x: x[1]['AUC'], reverse=True):
    print(f"  {d:20} {m['AUC']:>6.3f} {m['F1']:>6.3f} {m['Precision']:>6.3f} {m['Recall']:>6.3f} {m['Positives']:>6}")
print("-"*60)
print(f"  {'MEAN':20} {mean_auc:>6.3f} {mean_f1:>6.3f}")

# Save results
results = {'per_class': metrics, 'mean_auc': mean_auc, 'mean_f1': mean_f1}
json.dump(results, open(f"{RES_DIR}/test_eval_results.json", 'w'), indent=2)
print(f"\n Results saved to Drive")

# ROC Curves
fig, axes = plt.subplots(3, 5, figsize=(22, 13))
fig.patch.set_facecolor('#08090c')
for i, (d, m) in enumerate(metrics.items()):
    ax = axes.flatten()[i]
    fpr, tpr, _ = roc_curve(all_labels[:, DISEASES.index(d)],
                             all_probs[:, DISEASES.index(d)])
    ax.plot(fpr, tpr, color='#22d3a0', lw=1.5)
    ax.plot([0,1],[0,1], color='#1e2535', lw=1, ls='--')
    ax.fill_between(fpr, tpr, alpha=0.08, color='#22d3a0')
    ax.set_title(f"{d}\nAUC={m['AUC']:.3f}", color='white', fontsize=8)
    ax.set_facecolor('#08090c')
    ax.tick_params(colors='#5a6480', labelsize=7)
    for sp in ax.spines.values(): sp.set_color('#1e2535')
for j in range(len(metrics), 15): axes.flatten()[j].set_visible(False)
plt.suptitle(f'ROC Curves — Mean AUC={mean_auc:.3f} (Fine-Tuned DenseNet121)',
             color='white', fontsize=13)
plt.tight_layout()
plt.savefig(f"{RES_DIR}/roc_curves.png", dpi=150, facecolor='#08090c', bbox_inches='tight')
plt.show()
print("ROC curves saved")